# 하자 탐지 파이프라인 3단계 개선 — Before/After 비교

**스파이크/검증용 노트북입니다. 프로덕션 코드가 아닙니다.**

의류 하자(오염/구멍·찢어짐/얼룩) QC 파이프라인이 거쳐온 3단계 개선을, **같은 고정
이미지셋**에 대해 Before/After로 비교한다. 발표 자료용 그림·표는 `results/`에 저장한다.

| 단계 | Before -> After |
| --- | --- |
| ① 백본 교체 | CLIPSeg(`CIDAS/clipseg-rd64-refined`) -> SAM3(`facebook/sam3`) |
| ② 의류 세그멘테이션 도입 | 배경(바닥 등) 포함 전체 이미지 탐지 -> 의류 마스크로 사후 필터(`filter_by_garment`) |
| ③ 양성판정(오탐 후처리) 도입 | 필터 전 -> 주름·그림자·솔기 등 정상 패턴 억제(`suppress_benign`) |

## ⚠️ 반드시 읽을 것 — 이 노트북의 한계

- **정답 라벨(ground truth)이 없다.** 하자 위치·픽셀에 대한 정답 bbox/mask/클래스 라벨셋이
  아직 없어서, mAP@0.5 / mIoU / Precision·Recall·F1 / confusion matrix는 **계산하지 않는다**
  (계산해도 근거 없는 숫자가 된다). 대신 라벨 없이도 계산 가능한 대체 지표를 쓴다:
  탐지 개수, 평균 confidence, 추론 시간(ms), 필터로 제거된 인스턴스 수/비율.
  → 실제 정답 라벨셋이 생기면 이 노트북에 mAP/mIoU/F1 섹션을 추가하는 게 다음 단계다.
- **GPU가 필요하다.** SAM3(fp16 ~1.7GB) + CLIPSeg(~586MB)를 순서대로 로드한다 — 8GB VRAM
  카드에서 **동시에 올리면 OOM 위험**이 있어(다른 QC 스파이크 노트북들과 같은 이유) 이
  노트북도 CLIPSeg를 먼저 쓰고 언로드한 뒤 SAM3를 로드하는 순서로 짰다(§전환 셀 참고).
- **실행은 사용자가 직접 한다.** 이 노트북은 로컬(RTX 4070 8GB) 또는 그에 준하는 GPU
  환경에서 실행한다. 실행 결과(그림·표 안의 실제 숫자)는 돌려본 사람이 채우는 것이고,
  이 문서 자체에는 추측으로 채운 숫자가 없다 — 출력 셀은 비어 있다.
- **CLIP(원조 OpenAI CLIP)이 아니라 CLIPSeg다.** 요청 배경에 "CLIP 기반 탐지"라고 되어
  있었는데, 이 리포의 실제 첫 번째 백엔드는 CLIPSeg(CLIP 인코더 위에 세그멘테이션 디코더를
  얹은 모델)였다 — `worker/models/clipseg/`가 프로덕션에서 실제로 쓰던 백엔드다.
- 프롬프트 세트·임계치·필터 로직은 `validation/QUAL/qc_shared.py`에서 온다 — 두 백엔드가
  같은 로직을 공유해야 비교가 성립하기 때문이다(그 파일 docstring 참고).

## 0-1. 설정 — CONFIG (경로·상수는 전부 여기에 모은다)

In [ ]:
import sys
import time
from pathlib import Path

import numpy as np
import pandas as pd
import torch

# 프롬프트·임계치·필터 로직은 qc_shared.py 에서 온다(SAM3/CLIPSeg 두 백엔드가 공유).
sys.path.insert(0, "..")
from qc_shared import (  # noqa: E402
    BACKEND_THRESHOLDS,
    BENIGN_CONTAINMENT,
    BENIGN_PROMPTS,
    DEFECT_PROMPTS,
    GARMENT_OVERLAP_MIN,
    GARMENT_PROMPT,
    SAM3_OUTPUT_DIR,
    SHARED_IMAGE_DIR,
    VERDICT_THRESHOLDS,
    decide_verdict,
    draw_numbered_boxes,
    filter_by_garment,
    list_images,
    pick_largest,
    probs_to_instances,
    renumber,
    suppress_benign,
    to_serializable,
    use_korean_font,
)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"device: {DEVICE}")
if DEVICE != "cuda":
    print("[주의] GPU 가 없습니다 — SAM3/CLIPSeg 둘 다 CPU 로도 돌지만 매우 느립니다.")

# 이 노트북 하나의 산출물(발표용 PNG + CSV)은 전부 여기 한 곳에 모은다.
RESULTS_DIR = Path("./results")
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# §A 는 SAM3 를 이 노트북에서 다시 돌리지 않고, sam3_qc_test.ipynb 가 미리 저장해 둔 결과를
# 읽는다(§A 사전조건 셀에서 존재 여부를 확인한다). 없으면 그 노트북을 먼저 끝까지 실행할 것.
print(f"SAM3 사전 결과 폴더 : {SAM3_OUTPUT_DIR}")
print(f"결과 저장 폴더      : {RESULTS_DIR.resolve()}")

## 0-2. 고정 테스트 이미지셋

**아래 셀에서 한 번만 정의하고, 이후 §A·§B·§C·§4 전부 이 `IMAGE_PATHS`를 그대로 재사용한다.**
단계마다 다른 샘플을 쓰면 Before/After 비교가 무효화되므로, 다른 셀에서 이 목록을 새로 만들지
않는다.

In [ ]:
IMAGE_PATHS = list_images(SHARED_IMAGE_DIR)
if not IMAGE_PATHS:
    raise FileNotFoundError(
        f"{SHARED_IMAGE_DIR} 에 이미지가 없습니다. "
        "sam3-condition-check/sam3_qc_test.ipynb §0-5 를 먼저 실행해 이미지를 채워 주세요."
    )


def load_image(path):
    from PIL import Image

    return Image.open(path).convert("RGB")


print(f"고정 이미지셋 {len(IMAGE_PATHS)}장 (이 순서·이 파일들로 §A~§4 전부 비교한다):")
for _i, _p in enumerate(IMAGE_PATHS):
    print(f"  [{_i}] {_p.relative_to(SHARED_IMAGE_DIR)}")

## 0-3. 시각화 유틸

- `draw_before_after_boxes`: 살아남은(kept) 인스턴스는 기존 `draw_numbered_boxes`와 같은
  빨간 번호 박스로, **제거된(dropped) 인스턴스는 회색 박스 + X 표시**로 그린다 — 오탐이
  줄어드는 걸 한 장에서 바로 보여주기 위해서다.
- `save_fig`: 300dpi로 저장하고 실제 픽셀 크기를 출력해, 슬라이드에 바로 캡처해 쓸 수 있는
  해상도(가로 1600px 이상)인지 바로 확인할 수 있게 한다.
- 한글 폰트는 `qc_shared.use_korean_font()`를 그대로 쓴다 — 이 리포의 다른 QC 노트북들이
  전부 이 함수로 맑은 고딕(Windows 개발 환경)을 잡고 있어(Pretendard/NanumGothic 후보도
  포함) 새 폰트 의존성을 따로 추가하지 않는다. 발표 환경에 Pretendard를 쓰고 싶다면
  `qc_shared.use_korean_font()`의 후보 튜플에 `"Pretendard"`만 추가하면 된다.

In [ ]:
import matplotlib.pyplot as plt

print(f"matplotlib 한글 폰트: {use_korean_font()}")


def draw_before_after_boxes(pil_image, kept, dropped, dropped_color=(140, 140, 140)):
    """kept 는 draw_numbered_boxes 와 같은 빨간 번호 박스, dropped 는 회색 박스 + X."""
    import cv2
    from PIL import Image as PILImage

    canvas = np.array(pil_image).copy()
    for inst in dropped:
        x1, y1, x2, y2 = (int(v) for v in inst["bbox"])
        cv2.rectangle(canvas, (x1, y1), (x2, y2), dropped_color, 3)
        cv2.line(canvas, (x1, y1), (x2, y2), dropped_color, 3)
        cv2.line(canvas, (x1, y2), (x2, y1), dropped_color, 3)

    # kept 는 회색 X 를 그린 캔버스 위에 기존 함수로 덧그린다(빨간 번호 박스 스타일 재사용).
    return draw_numbered_boxes(PILImage.fromarray(canvas), kept)


def save_fig(fig, filename):
    path = RESULTS_DIR / filename
    fig.savefig(path, dpi=300, bbox_inches="tight")
    w_in, h_in = fig.get_size_inches()
    print(f"저장: {path}  (약 {w_in * 300:.0f}x{h_in * 300:.0f}px @300dpi)")
    return path

## 0-4. 지표 안내 (다시 한번)

정답 라벨이 없어 아래 지표만 쓴다 — 전부 라벨 없이 계산 가능하다:

| 지표 | 어디서 |
| --- | --- |
| 탐지 개수(백엔드/단계별) | §A·§B·§C 전부 |
| 평균 confidence | §A |
| 추론 시간(ms) | §A(CLIPSeg 실측) · §4(SAM3 실측, §B에서 잰 값 재사용) |
| 필터로 제거된 인스턴스 수/비율 | §B(배경 제외), §C(양성 억제) |
| 타입별 억제 카운트 (confusion matrix 대체) | §C |

---

## A. 백본 교체 효과 — CLIPSeg → SAM3

**무엇을 바꿨는가**: 하자 탐지 백엔드만 CLIPSeg → SAM3로 바꾼다.
**무엇이 통제됐는가**: 프롬프트 세트(`qc_shared.ACTIVE_PROMPT_SET`), 의류 필터·양성 억제
로직(`filter_by_garment`/`suppress_benign`), 판정 임계 구간(`VERDICT_THRESHOLDS`)은 두
백엔드가 `qc_shared.py` 하나를 공유하므로 동일하다.

**⚠️ 이 비교에서 통제되지 않은 변수**: 두 백엔드의 confidence 스케일 자체가 다르다
(CLIPSeg는 픽셀 sigmoid 확률, SAM3는 `pred_logits.sigmoid() * presence_logits.sigmoid()`).
탐지 임계치도 백엔드별로 다르게 잡혀 있다(`BACKEND_THRESHOLDS` — CLIPSeg 0.45 / SAM3
0.30, qc_shared.py 주석 참고: "003 계획의 교훈 — 모델마다 점수 스케일이 달라 고정 임계치를
모델 무관하게 재사용하면 안 된다"). 즉 **백본 차이와 임계치 차이가 결과 차이에 각각 얼마나
기여했는지는 이 비교만으로 분리되지 않는다** — 두 축을 분리하려면 같은 confidence 분포에서
임계치만 스윕하는 별도 실험이 필요하다.

### A-0. 사전조건 확인

이 섹션은 SAM3 쪽 결과를 **이 노트북에서 다시 추론하지 않고**, `sam3_qc_test.ipynb`가
미리 저장해 둔 `outputs/sam3_qc_test/<파일명>_qc.json` / `_boxes.png`를 읽는다. SAM3와
CLIPSeg를 한 커널에 동시에 올리는 위험을 피하기 위해서다(§전환 셀 참고). 없으면 아래
셀이 무엇을 먼저 실행해야 하는지 알려준다.

In [ ]:
_missing_sam3 = [
    p.name
    for p in IMAGE_PATHS
    if not (SAM3_OUTPUT_DIR / f"{p.stem}_qc.json").exists()
    or not (SAM3_OUTPUT_DIR / f"{p.stem}_boxes.png").exists()
]

if _missing_sam3:
    print(f"[알림] SAM3 사전 결과가 없는 이미지 {len(_missing_sam3)}장: {_missing_sam3}")
    print("       sam3-condition-check/sam3_qc_test.ipynb 를 §5·§9(넘버링 시각화 + 배치 실행)")
    print("       까지 끝까지 실행하면 두 파일이 outputs/sam3_qc_test/ 에 생성됩니다.")
else:
    print(f"SAM3 사전 결과 확인 완료 — {len(IMAGE_PATHS)}장 전부 존재합니다.")

### A-1. CLIPSeg 모델 로드

In [ ]:
from transformers import CLIPSegForImageSegmentation, CLIPSegProcessor

CLIPSEG_MODEL_ID = "CIDAS/clipseg-rd64-refined"
CLIPSEG_INPUT_SIZE = 224  # worker/models/clipseg/segmenter.py 와 같은 값

if DEVICE == "cuda":
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()

try:
    clipseg_processor = CLIPSegProcessor.from_pretrained(CLIPSEG_MODEL_ID)
    clipseg_model = CLIPSegForImageSegmentation.from_pretrained(CLIPSEG_MODEL_ID).to(DEVICE).eval()
    print(f"CLIPSeg 로드 완료: {CLIPSEG_MODEL_ID}  device={DEVICE}")
except Exception as exc:
    print(f"[모델 로드 실패] {type(exc).__name__}: {exc}")
    raise

### A-2. CLIPSeg 추론 — `clipseg-condition-check/clipseg_qc_test.ipynb` §1~§3과 동일 로직

새로 만든 로직이 아니다 — 기존 CLIPSeg 검증 노트북의 `clipseg_probs`/`detect_all`/
`inspect_image`를 그대로 옮기고 이름만 `clipseg_` 접두어를 붙였다(뒤에서 SAM3 쪽 함수와
이름이 겹치지 않게).

In [ ]:
CLIPSEG_PROB_THRESHOLD = BACKEND_THRESHOLDS["clipseg"]["prob"]
CLIPSEG_GARMENT_PROB_THRESHOLD = BACKEND_THRESHOLDS["clipseg"]["garment_prob"]


def clipseg_probs(pil_image, prompts):
    """프롬프트 리스트 -> (n, H, W) 확률맵 (원본 해상도)."""
    text_inputs = clipseg_processor.tokenizer(list(prompts), return_tensors="pt", padding=True)
    image_inputs = clipseg_processor.image_processor(
        [pil_image] * len(prompts),
        size={"height": CLIPSEG_INPUT_SIZE, "width": CLIPSEG_INPUT_SIZE},
        return_tensors="pt",
    )
    inputs = {**text_inputs, "pixel_values": image_inputs["pixel_values"]}
    inputs = {k: v.to(DEVICE) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = clipseg_model(**inputs)

    logits = outputs.logits
    if logits.ndim == 2:  # 프롬프트가 1개면 배치 차원이 사라진다
        logits = logits.unsqueeze(0)

    probs = torch.sigmoid(logits).unsqueeze(1)
    probs = torch.nn.functional.interpolate(
        probs, size=(pil_image.height, pil_image.width), mode="bilinear", align_corners=False,
    )
    return probs[:, 0].detach().cpu().numpy()


def clipseg_detect_all(pil_image, prompts=DEFECT_PROMPTS, threshold=CLIPSEG_PROB_THRESHOLD):
    tags = list(prompts)
    maps = clipseg_probs(pil_image, [prompts[t] for t in tags])
    merged = []
    for tag, prob_map in zip(tags, maps):
        merged.extend(probs_to_instances(prob_map, tag, prompts[tag], threshold))
    return renumber(merged)


def clipseg_detect_garment(pil_image, threshold=CLIPSEG_GARMENT_PROB_THRESHOLD):
    prob_map = clipseg_probs(pil_image, [GARMENT_PROMPT])[0]
    return probs_to_instances(prob_map, "GARMENT", GARMENT_PROMPT, threshold)


def clipseg_benign_instances(pil_image, threshold=CLIPSEG_PROB_THRESHOLD):
    tags = list(BENIGN_PROMPTS)
    maps = clipseg_probs(pil_image, [BENIGN_PROMPTS[t] for t in tags])
    found = []
    for tag, prob_map in zip(tags, maps):
        found.extend(probs_to_instances(prob_map, tag, BENIGN_PROMPTS[tag], threshold))
    return found


def clipseg_inspect_image(pil_image):
    """CLIPSeg 전체 파이프라인(의류 필터 + 양성 억제 모두 적용) — 지금 프로덕션 후보와
    같은 조건으로 SAM3 와 비교하기 위해 둘 다 켠 상태로 고정한다."""
    raw = clipseg_detect_all(pil_image)
    picked = pick_largest(clipseg_detect_garment(pil_image))
    garment_mask = picked["mask"] if picked is not None else None
    benigns = clipseg_benign_instances(pil_image)
    after_garment, off_garment = filter_by_garment(raw, garment_mask, GARMENT_OVERLAP_MIN)
    final, on_benign = suppress_benign(after_garment, benigns, BENIGN_CONTAINMENT)
    trace = {"raw": raw, "off_garment": off_garment, "on_benign": on_benign, "final": final}
    return renumber(final), trace


clipseg_results = {}
clipseg_traces = {}
clipseg_timing_ms = {}
print(f"CLIPSeg 추론 시작 — 고정 이미지 {len(IMAGE_PATHS)}장")
for _path in IMAGE_PATHS:
    _img = load_image(_path)
    _t0 = time.perf_counter()
    _final, _trace = clipseg_inspect_image(_img)
    clipseg_timing_ms[_path] = (time.perf_counter() - _t0) * 1000
    clipseg_results[_path], clipseg_traces[_path] = _final, _trace
    print(f"  {_path.name:<40} {clipseg_timing_ms[_path]:>7.1f}ms  {len(_final):>3}건")

### A-3. 이미지 하나씩 Before(CLIPSeg) | After(SAM3) 비교

발표 자료용이라 전체를 한 Figure에 몰지 않고 **이미지 1장당 별도 Figure**를 만든다.

In [ ]:
import json

import matplotlib.image as mpimg

for _path in IMAGE_PATHS:
    _boxes_png = SAM3_OUTPUT_DIR / f"{_path.stem}_boxes.png"
    _qc_json = SAM3_OUTPUT_DIR / f"{_path.stem}_qc.json"
    if not _boxes_png.exists() or not _qc_json.exists():
        print(f"[건너뜀] {_path.name}: SAM3 사전 결과 없음 (A-0 참고)")
        continue

    _sam3_payload = json.loads(_qc_json.read_text(encoding="utf-8"))
    _sam3_insts = _sam3_payload.get("instances", [])

    _clipseg_view = draw_numbered_boxes(load_image(_path), clipseg_results[_path])
    _sam3_view = mpimg.imread(_boxes_png)

    fig, axes = plt.subplots(1, 2, figsize=(16, 7))
    axes[0].imshow(_clipseg_view)
    axes[0].set_title(
        f"Before: CLIPSeg — {len(clipseg_results[_path])}건, {clipseg_timing_ms[_path]:.0f}ms"
    )
    axes[0].axis("off")
    axes[1].imshow(_sam3_view)
    axes[1].set_title(f"After: SAM3 — {len(_sam3_insts)}건 (사전 실행 결과, {_path.stem}_qc.json)")
    axes[1].axis("off")
    plt.suptitle(_path.name)
    plt.tight_layout()
    save_fig(fig, f"A_backbone_{_path.stem}.png")
    plt.show()

### A-4. 정량 지표

In [ ]:
_rows = []
for _path in IMAGE_PATHS:
    _qc_json = SAM3_OUTPUT_DIR / f"{_path.stem}_qc.json"
    _sam3_insts = (
        json.loads(_qc_json.read_text(encoding="utf-8"))["instances"] if _qc_json.exists() else None
    )
    _row = {
        "image": _path.name,
        "clipseg_count": len(clipseg_results[_path]),
        "sam3_count": (len(_sam3_insts) if _sam3_insts is not None else None),
        "clipseg_avg_conf": (
            round(float(np.mean([d["confidence"] for d in clipseg_results[_path]])), 4)
            if clipseg_results[_path]
            else 0.0
        ),
        "sam3_avg_conf": (
            round(float(np.mean([d["confidence"] for d in _sam3_insts])), 4)
            if _sam3_insts
            else (0.0 if _sam3_insts is not None else None)
        ),
        "clipseg_time_ms": round(clipseg_timing_ms[_path], 1),
    }
    for _type in DEFECT_PROMPTS:
        _row[f"clipseg_{_type}"] = sum(1 for d in clipseg_results[_path] if d["type"] == _type)
        _row[f"sam3_{_type}"] = (
            sum(1 for d in _sam3_insts if d["type"] == _type) if _sam3_insts is not None else None
        )
    _rows.append(_row)

section_a_df = pd.DataFrame(_rows)
print(section_a_df.to_string(index=False))
section_a_df.to_csv(RESULTS_DIR / "section_a_backbone_metrics.csv", index=False, encoding="utf-8-sig")
print()
print("(SAM3 쪽 추론 시간은 이 노트북에서 SAM3 를 아직 로드하지 않아 여기 없다 — §4 종합 표에서")
print(" §B 로드 시점에 잰 값을 함께 보여준다.)")
section_a_df

### A-5. 실패 사례 — 두 백엔드가 갈리거나 둘 다 놓치는 경우

아래 셀은 "verdict가 서로 다른" 또는 "둘 다 0건"인 후보를 자동으로 찾아준다. **최소 1장은
직접 이미지를 열어 확인하고, 왜 실패하는지(예: 찢어짐이 너무 미세해서 둘 다 못 잡음)를
이 markdown 셀에 이어서 적을 것** — "좋아진 사례"만 보여주면 과장이 된다.

> (실행 후 채울 것) 예: `[이미지명]` — CLIPSeg는 X건/SAM3는 Y건 탐지했지만 둘 다 실제
> 하자 Z를 놓쳤다. 원인 추정: ...

In [ ]:
print("실패 사례 후보 (verdict 불일치 또는 두 백엔드 모두 탐지 0건):")
_found_any = False
for _path in IMAGE_PATHS:
    _qc_json = SAM3_OUTPUT_DIR / f"{_path.stem}_qc.json"
    if not _qc_json.exists():
        continue
    _sam3_payload = json.loads(_qc_json.read_text(encoding="utf-8"))
    _sam3_insts = _sam3_payload.get("instances", [])
    _clip_v, _ = decide_verdict(clipseg_results[_path], VERDICT_THRESHOLDS)
    _sam3_v = _sam3_payload.get("verdict")
    _both_zero = len(clipseg_results[_path]) == 0 and len(_sam3_insts) == 0
    if _clip_v != _sam3_v or _both_zero:
        _found_any = True
        print(f"  - {_path.name}: clipseg={_clip_v} vs sam3={_sam3_v} (둘 다 0건={_both_zero})")
if not _found_any:
    print("  (이번 고정 이미지셋에서는 자동 기준으로 걸린 후보가 없습니다 — 그래도 위 A-3 그림을")
    print("   직접 훑어 confidence가 애매하게 낮은 사례를 최소 1장 찾아 위 markdown에 적을 것)")

---

## 모델 전환 — CLIPSeg 언로드 → SAM3 로드

8GB VRAM 카드에서 SAM3 + CLIPSeg 를 동시에 올리면 OOM 위험이 있다(다른 QC 스파이크
노트북의 실측 경고와 동일한 이유). §B·§C는 SAM3만 있으면 되므로 CLIPSeg를 먼저 내린다.

In [ ]:
clipseg_model = None
clipseg_processor = None
if DEVICE == "cuda":
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()
print("CLIPSeg 언로드 완료. SAM3 로드를 시작합니다.")

import transformers

try:
    from transformers import Sam3Model, Sam3Processor
except ImportError:
    print("[에러] SAM3 클래스를 임포트하지 못했습니다.")
    print(f"  현재 transformers : {transformers.__version__}  (필요: >= 5.0)")
    raise

SAM3_MODEL_ID = "facebook/sam3"
SAM3_DTYPE = torch.float16

try:
    sam3_processor = Sam3Processor.from_pretrained(SAM3_MODEL_ID)
    sam3_model = Sam3Model.from_pretrained(SAM3_MODEL_ID, dtype=SAM3_DTYPE).to(DEVICE).eval()
    print(f"SAM3 로드 완료: {SAM3_MODEL_ID}  dtype={SAM3_DTYPE}  device={DEVICE}")
except Exception as exc:
    _msg = str(exc).lower()
    if any(k in _msg for k in ("gated", "401", "403", "awaiting", "access to model")):
        print("[게이트 접근 실패] facebook/sam3 는 gated repo 입니다.")
        print("  sam3-condition-check/sam3_qc_test.ipynb §0-1(HF 로그인)을 먼저 실행하세요.")
    else:
        print(f"[모델 로드 실패] {type(exc).__name__}: {exc}")
    raise

---

## B. 의류 세그멘테이션(사후 필터) 도입 효과

**무엇을 바꿨는가**: `RESTRICT_TO_GARMENT`만 False → True로 바꾼다. False일 때는 전체
이미지에서 그대로 하자 프롬프트를 돌리고, True일 때는 별도로 구한 "clothing" 마스크와
겹치지 않는 인스턴스를 사후에 걸러낸다(`filter_by_garment`) — 배경(바닥·소품)에서 잡히는
오탐을 없애기 위한 단계다.

**무엇이 통제됐는가**: 이 비교에서 크롭(`CROP_BEFORE_DETECT`, 의류 bbox로 이미지 자체를
잘라서 그 안에서만 탐지하는 방식)은 **두 조건 다 False로 고정**한다. 크롭은 별도로 실측한
결과 상품 사진 8장 중 6장에서 작거나 애매한 하자를 놓치는 회귀를 냈던 전례가 있어
(`qc_shared.py` 주석 참고) 기본값이 꺼져 있다 — 여기 섞으면 "가먼트 필터 효과"와 "크롭
자체의 부작용"이 뒤섞여 원인을 분리할 수 없다. 양성 억제(`SUPPRESS_BENIGN`)도 두 조건 다
**False로 고정**해 §C 효과와 섞이지 않게 통제한다.

**⚠️ 통제되지 않은 변수**: 없음(단일 변수 실험 — `restrict_to_garment` 하나만 바뀐다).

In [ ]:
SAM3_SCORE_THRESHOLD = BACKEND_THRESHOLDS["sam3"]["score"]
SAM3_GARMENT_SCORE_THRESHOLD = BACKEND_THRESHOLDS["sam3"]["garment_score"]
SAM3_MASK_THRESHOLD = 0.5


def _sam3_to_device(inputs):
    inputs = inputs.to(DEVICE)
    if "pixel_values" in inputs and inputs["pixel_values"].dtype.is_floating_point:
        inputs["pixel_values"] = inputs["pixel_values"].to(SAM3_DTYPE)
    return inputs


def sam3_detect(pil_image, prompt, *, score_threshold=SAM3_SCORE_THRESHOLD):
    inputs = _sam3_to_device(sam3_processor(images=pil_image, text=prompt, return_tensors="pt"))
    with torch.no_grad():
        outputs = sam3_model(**inputs)
    result = sam3_processor.post_process_instance_segmentation(
        outputs,
        threshold=score_threshold,
        mask_threshold=SAM3_MASK_THRESHOLD,
        target_sizes=inputs["original_sizes"].tolist(),
    )[0]
    return result


def sam3_detect_all(pil_image, prompts=DEFECT_PROMPTS, score_threshold=SAM3_SCORE_THRESHOLD):
    merged = []
    for defect_type, prompt in prompts.items():
        res = sam3_detect(pil_image, prompt, score_threshold=score_threshold)
        for score, box, mask in zip(res["scores"].tolist(), res["boxes"].tolist(), res["masks"]):
            merged.append(
                {
                    "type": defect_type,
                    "prompt": prompt,
                    "bbox": [int(round(v)) for v in box],
                    "confidence": round(float(score), 4),
                    "mask": mask.detach().cpu().numpy().astype(np.uint8),
                }
            )
    return renumber(merged)


def sam3_detect_garment(pil_image):
    res = sam3_detect(pil_image, GARMENT_PROMPT, score_threshold=SAM3_GARMENT_SCORE_THRESHOLD)
    found = []
    for score, box, mask in zip(res["scores"].tolist(), res["boxes"].tolist(), res["masks"]):
        found.append(
            {
                "type": "GARMENT",
                "confidence": round(float(score), 4),
                "bbox": [int(round(v)) for v in box],
                "mask": mask.detach().cpu().numpy().astype(np.uint8),
            }
        )
    return found


def sam3_detect_benign(pil_image):
    found = []
    for tag, prompt in BENIGN_PROMPTS.items():
        res = sam3_detect(pil_image, prompt)
        for score, box, mask in zip(res["scores"].tolist(), res["boxes"].tolist(), res["masks"]):
            found.append(
                {
                    "type": tag,
                    "confidence": round(float(score), 4),
                    "bbox": [int(round(v)) for v in box],
                    "mask": mask.detach().cpu().numpy().astype(np.uint8),
                }
            )
    return found


def sam3_inspect_image(pil_image, *, restrict_to_garment, apply_benign_suppression):
    """§B·§C 실험 전용 — restrict_to_garment/apply_benign_suppression 을 인자로 명시해
    호출부에서 조건이 항상 눈에 보이게 한다(모듈 상수에 암묵적으로 기대지 않는다).
    크롭(crop-then-detect)은 이 실험에서 항상 하지 않는다(§B markdown 참고).
    """
    raw = sam3_detect_all(pil_image)
    garment_mask = None
    if restrict_to_garment:
        picked = pick_largest(sam3_detect_garment(pil_image))
        garment_mask = picked["mask"] if picked is not None else None
    benigns = sam3_detect_benign(pil_image) if apply_benign_suppression else []
    after_garment, off_garment = filter_by_garment(raw, garment_mask, GARMENT_OVERLAP_MIN)
    final, on_benign = suppress_benign(after_garment, benigns, BENIGN_CONTAINMENT)
    trace = {
        "raw": raw,
        "off_garment": off_garment,
        "on_benign": on_benign,
        "final": final,
        "garment_found": garment_mask is not None,
    }
    return renumber(final), trace


sam3_before_results, sam3_before_traces = {}, {}
sam3_after_results, sam3_after_traces = {}, {}
sam3_timing_ms = {}

print(f"SAM3 §B 추론 시작 — 고정 이미지 {len(IMAGE_PATHS)}장 (의류 필터 off -> on)")
for _path in IMAGE_PATHS:
    _img = load_image(_path)

    _t0 = time.perf_counter()
    _final_off, _trace_off = sam3_inspect_image(
        _img, restrict_to_garment=False, apply_benign_suppression=False
    )
    # 대표 SAM3 추론시간 — 이 시점(§4 종합 표)에서 재사용한다. 워밍업 없이 실측이라
    # 첫 이미지는 다소 느릴 수 있다(로드 직후 첫 forward 비용).
    sam3_timing_ms[_path] = (time.perf_counter() - _t0) * 1000
    sam3_before_results[_path], sam3_before_traces[_path] = _final_off, _trace_off

    _final_on, _trace_on = sam3_inspect_image(
        _img, restrict_to_garment=True, apply_benign_suppression=False
    )
    sam3_after_results[_path], sam3_after_traces[_path] = _final_on, _trace_on

    print(
        f"  {_path.name:<40} off={len(_final_off):>3}건 -> on={len(_final_on):>3}건 "
        f"(배경으로 제외 {len(_trace_on['off_garment']):>3}건)"
    )

### B-1. 이미지 하나씩 Before(필터 없음) | After(의류 필터 적용) 비교

In [ ]:
for _path in IMAGE_PATHS:
    _img = load_image(_path)
    _before_view = draw_numbered_boxes(_img, sam3_before_results[_path])
    _after_view = draw_before_after_boxes(
        _img, sam3_after_results[_path], sam3_after_traces[_path]["off_garment"]
    )

    fig, axes = plt.subplots(1, 2, figsize=(16, 7))
    axes[0].imshow(_before_view)
    axes[0].set_title(f"Before: 의류 필터 없음 — {len(sam3_before_results[_path])}건 (배경 포함)")
    axes[0].axis("off")
    axes[1].imshow(_after_view)
    axes[1].set_title(
        f"After: 의류 필터 적용 — {len(sam3_after_results[_path])}건 유지, "
        f"{len(sam3_after_traces[_path]['off_garment'])}건 배경으로 제외(회색 X)"
    )
    axes[1].axis("off")
    plt.suptitle(_path.name)
    plt.tight_layout()
    save_fig(fig, f"B_garment_filter_{_path.stem}.png")
    plt.show()

### B-2. 정량 지표

In [ ]:
_rows = []
for _path in IMAGE_PATHS:
    _before_n = len(sam3_before_results[_path])
    _t = sam3_after_traces[_path]
    _rows.append(
        {
            "image": _path.name,
            "raw_count": _before_n,
            "off_garment_removed": len(_t["off_garment"]),
            "off_garment_removed_pct": round(100 * len(_t["off_garment"]) / max(1, _before_n), 1),
            "final_count": len(_t["final"]),
            "garment_found": _t["garment_found"],
        }
    )
section_b_df = pd.DataFrame(_rows)
print(section_b_df.to_string(index=False))
section_b_df.to_csv(RESULTS_DIR / "section_b_garment_filter_metrics.csv", index=False, encoding="utf-8-sig")
section_b_df

### B-3. 실패 사례 — 의류를 못 찾아 필터가 no-op된 경우

`garment_found=False`면 "clothing" 마스크를 아예 못 찾아 이번 이미지에서는 필터가
사실상 아무 일도 하지 않은 것이다. 후보가 없더라도 위 그림 중 배경 오탐이 여전히
남아있는 사례가 있는지 최소 1장 확인해서 적을 것.

> (실행 후 채울 것)

In [ ]:
_no_garment = section_b_df[~section_b_df["garment_found"]]
if len(_no_garment):
    print("실패 후보(의류를 못 찾아 필터가 no-op됨):")
    print(_no_garment.to_string(index=False))
else:
    print("이번 고정 이미지셋에서는 의류 탐지 실패(garment_found=False) 사례가 없었습니다.")

---

## C. 양성판정(오탐 후처리) 도입 효과

**무엇을 바꿨는가**: `apply_benign_suppression`만 False → True로 바꾼다. True면 주름·
접힘·그림자·솔기·프린트·암홀·넥라인·밑단·소매끝 같은 "정상인데 하자로 오인되는" 영역과
겹치는 하자 인스턴스를 억제한다(`suppress_benign`, 로직 상세는 `qc_shared.py` 참고 —
구조적 개구부는 겹치기만 하면 즉시 억제, 질감형은 confidence 비교 후에만 억제).

**무엇이 통제됐는가**: `restrict_to_garment=True`로 **두 조건 다 고정**한다(§B 결과를
그대로 이어받는다) — 그래야 §B 효과와 섞이지 않고 §C만의 순수 효과를 본다.

**⚠️ 통제되지 않은 변수**: 없음(단일 변수 실험 — `apply_benign_suppression` 하나만 바뀐다).

In [ ]:
sam3_c_before_results, sam3_c_before_traces = {}, {}
sam3_c_after_results, sam3_c_after_traces = {}, {}

print(f"SAM3 §C 추론 시작 — 고정 이미지 {len(IMAGE_PATHS)}장 (양성 억제 off -> on)")
for _path in IMAGE_PATHS:
    _img = load_image(_path)
    _final_off, _trace_off = sam3_inspect_image(
        _img, restrict_to_garment=True, apply_benign_suppression=False
    )
    _final_on, _trace_on = sam3_inspect_image(
        _img, restrict_to_garment=True, apply_benign_suppression=True
    )
    sam3_c_before_results[_path], sam3_c_before_traces[_path] = _final_off, _trace_off
    sam3_c_after_results[_path], sam3_c_after_traces[_path] = _final_on, _trace_on
    print(
        f"  {_path.name:<40} off={len(_final_off):>3}건 -> on={len(_final_on):>3}건 "
        f"(양성으로 억제 {len(_trace_on['on_benign']):>3}건)"
    )

### C-1. 이미지 하나씩 Before(억제 없음) | After(양성 억제 적용) 비교

In [ ]:
for _path in IMAGE_PATHS:
    _img = load_image(_path)
    _before_view = draw_numbered_boxes(_img, sam3_c_before_results[_path])
    _after_view = draw_before_after_boxes(
        _img, sam3_c_after_results[_path], sam3_c_after_traces[_path]["on_benign"]
    )

    fig, axes = plt.subplots(1, 2, figsize=(16, 7))
    axes[0].imshow(_before_view)
    axes[0].set_title(f"Before: 양성 억제 없음 — {len(sam3_c_before_results[_path])}건")
    axes[0].axis("off")
    axes[1].imshow(_after_view)
    axes[1].set_title(
        f"After: 양성 억제 적용 — {len(sam3_c_after_results[_path])}건 유지, "
        f"{len(sam3_c_after_traces[_path]['on_benign'])}건 억제(회색 X)"
    )
    axes[1].axis("off")
    plt.suptitle(_path.name)
    plt.tight_layout()
    save_fig(fig, f"C_benign_suppression_{_path.stem}.png")
    plt.show()

### C-2. 정량 지표 — 오탐 감소율 + 타입별 억제 카운트

정답 라벨이 없어 Precision/Recall/F1/confusion matrix 대신, **억제된 인스턴스가 어떤
정상 패턴(주름/그림자/솔기 등)에 의해 몇 건씩 억제됐는지**를 카운트 표로 대신한다 —
"이 필터가 정확히 무엇을 걷어냈는지"를 라벨 없이도 보여줄 수 있는 대체 지표다.

In [ ]:
_rows = []
_type_suppress_rows = []
for _path in IMAGE_PATHS:
    _before_n = len(sam3_c_before_results[_path])
    _dropped = sam3_c_after_traces[_path]["on_benign"]
    _after_n = len(sam3_c_after_results[_path])
    _rows.append(
        {
            "image": _path.name,
            "before_count": _before_n,
            "suppressed_count": len(_dropped),
            "suppressed_pct": round(100 * len(_dropped) / max(1, _before_n), 1),
            "after_count": _after_n,
        }
    )
    for _d in _dropped:
        _type_suppress_rows.append(
            {
                "image": _path.name,
                "defect_type": _d["type"],
                "suppressed_by": _d.get("suppressed_by", "?"),
            }
        )

section_c_df = pd.DataFrame(_rows)
print(section_c_df.to_string(index=False))
section_c_df.to_csv(RESULTS_DIR / "section_c_benign_suppression_metrics.csv", index=False, encoding="utf-8-sig")

type_suppress_df = pd.DataFrame(_type_suppress_rows)
print()
print("타입별 억제 상세 (confusion matrix 대체):")
if len(type_suppress_df):
    print(
        type_suppress_df.groupby("defect_type")
        .size()
        .rename("suppressed_count")
        .reset_index()
        .to_string(index=False)
    )
    type_suppress_df.to_csv(RESULTS_DIR / "section_c_suppressed_detail.csv", index=False, encoding="utf-8-sig")
else:
    print("(이번 고정 이미지셋에서는 억제된 인스턴스가 없었습니다)")
section_c_df

### C-3. 과도한 필터링 사례 — "정상까지 걸러버린" 케이스

억제 비율이 유독 높은 이미지를 자동으로 찾아 후보로 보여준다. **후보가 있든 없든, 위
그림들을 직접 보고 실제 하자가 억제된 사례가 있는지 최소 1장 확인해서 적을 것** — 이
로직(`suppress_benign`)은 라벨링된 검증셋 없이 잡은 임계치(`BENIGN_CONTAINMENT`)를 쓰고
있어(`qc_shared.py` 주석 참고), 과도한 억제가 실제로 있을 수 있다.

> (실행 후 채울 것)

In [ ]:
_OVER_FILTER_THRESHOLD_PCT = 50.0
_over_filtered = section_c_df[section_c_df["suppressed_pct"] >= _OVER_FILTER_THRESHOLD_PCT]
if len(_over_filtered):
    print(f"과도한 필터링 의심 후보(원래 탐지의 {_OVER_FILTER_THRESHOLD_PCT:.0f}% 이상 억제):")
    print(_over_filtered.to_string(index=False))
else:
    print(f"이번 셋에서 {_OVER_FILTER_THRESHOLD_PCT:.0f}% 이상 억제된 이미지는 없었습니다.")

---

## 4. 종합

3단계를 관통하는 대표 이미지를 골라 **원본 → CLIPSeg → SAM3+의류필터 → 최종(+양성판정)**
4단 그리드로 나열하고, 단계별 핵심 지표를 하나의 표로 정리한다.

In [ ]:
# 발표에서 임팩트가 큰 이미지로 바꿔서 쓸 것 — §A-5/§B-3/§C-3 에서 찾은 사례를 넣어도 좋다.
SUMMARY_INDICES = [0]

for _idx in SUMMARY_INDICES:
    _path = IMAGE_PATHS[_idx]
    _img = load_image(_path)

    _panel_original = np.array(_img)
    _panel_clipseg = draw_numbered_boxes(_img, clipseg_results[_path])
    _panel_garment = draw_numbered_boxes(_img, sam3_after_results[_path])  # §B After (필터만)
    _panel_final = draw_numbered_boxes(_img, sam3_c_after_results[_path])  # §C After (필터+억제)

    fig, axes = plt.subplots(1, 4, figsize=(22, 6))
    _titles = [
        "원본",
        f"① CLIPSeg\n{len(clipseg_results[_path])}건",
        f"② +의류 필터(SAM3)\n{len(sam3_after_results[_path])}건",
        f"③ +양성판정(최종)\n{len(sam3_c_after_results[_path])}건",
    ]
    for ax, img, title in zip(axes, [_panel_original, _panel_clipseg, _panel_garment, _panel_final], _titles):
        ax.imshow(img)
        ax.set_title(title)
        ax.axis("off")
    plt.suptitle(f"{_path.name} — 3단계 개선 종합")
    plt.tight_layout()
    save_fig(fig, f"summary_4panel_{_path.stem}.png")
    plt.show()

In [ ]:
_rows = []
for _path in IMAGE_PATHS:
    _rows.append(
        {
            "image": _path.name,
            "clipseg_count": len(clipseg_results[_path]),
            "clipseg_time_ms": round(clipseg_timing_ms[_path], 1),
            "sam3_before_garment_count": len(sam3_before_results[_path]),
            "sam3_after_garment_count": len(sam3_after_results[_path]),
            "sam3_final_count": len(sam3_c_after_results[_path]),
            "sam3_time_ms": round(sam3_timing_ms[_path], 1),
            "off_garment_removed": len(sam3_after_traces[_path]["off_garment"]),
            "on_benign_removed": len(sam3_c_after_traces[_path]["on_benign"]),
        }
    )

summary_df = pd.DataFrame(_rows)
print(summary_df.to_string(index=False))
summary_df.to_csv(RESULTS_DIR / "summary_all_stages.csv", index=False, encoding="utf-8-sig")
print()
print(f"저장 완료 -> {RESULTS_DIR.resolve()}")
print("(sam3_time_ms 는 §B 에서 의류 필터 off 상태로 잰 1회 추론시간이다 — 웜업 없이 첫 실행 기준)")
summary_df

---

## 결론 (실행 후 채울 것)

| # | 확인 항목 | 근거 셀 | 결과 |
| --- | --- | --- | --- |
| 1 | 백본 교체로 탐지 개수/평균 confidence가 실제로 개선됐는가 | §A-4 | |
| 2 | 의류 필터가 배경(바닥) 오탐을 실제로 줄였는가 (몇 %) | §B-2 | |
| 3 | 양성판정 로직의 오탐 감소율은 얼마인가 | §C-2 | |
| 4 | 과도한 필터링(정상 하자까지 억제)이 관찰됐는가 | §C-3 | |
| 5 | 두 백엔드 모두 실패하는 케이스는 어떤 유형인가 | §A-5 | |
| 6 | 추론 시간 트레이드오프는 감당할 만한가 | §4 요약표 | |

### 한계 (감추지 않고 남겨둔다)

- 정답 라벨이 없어 mAP/mIoU/Precision·Recall·F1/confusion matrix는 계산하지 않았다 —
  위 표들은 전부 라벨 없이 계산 가능한 대체 지표다.
- §A(백본 비교)는 confidence 스케일과 임계치가 백엔드마다 달라, 백본 자체의 차이와
  임계치 차이가 섞여 있다(§A 마크다운 참고).
- 고정 이미지셋 크기(`len(IMAGE_PATHS)`)가 작으면 위 비율(%)들은 표본이 작아 노이즈가
  클 수 있다 — 이미지 수가 적었다면 결론에 그 사실을 함께 적을 것.